# Per-Cell-Line CyEmbed Archetypes and Cross-Line Matching

**Goal.** Fit `CyEmbed` **independently on each of the five breast cancer cell lines**, choose an
appropriate number of archetypes per line, and then ask which archetypes **correspond across lines**.

This is the complement to the joint analysis (`CellLines_CyEmbed_JointAnalysis.ipynb`). Instead of
forcing one shared basis on all lines and correcting for line baselines with an offset term, each
line gets its own basis and correspondence is established *after* fitting. That makes "is this state
shared?" an empirical result rather than a modelling assumption.

### Design decisions that shape everything downstream

1. **No sample offset.** Each run sees a single cell line, so a per-sample intercept
   $B_s$ is unidentifiable — it would be perfectly confounded with the archetype intercept.
   `use_sample_offset=False` throughout.

2. **One shared scaler, not five.** The z-score scaler is fitted **once** on all five lines
   (balanced, 5,000 cells/line — the same scaler the joint analysis uses) and then applied to each
   line. Fitting a separate scaler per line would put every line's archetypes in its own private
   units and make cross-line comparison meaningless. With a shared scaler, an archetype from MCF7
   and one from HCC70 live in the same 29-dimensional space.

3. **$K$ is chosen per line, and lines may differ.** Nothing requires HCC70 and MCF7 to need the same
   number of states. Selection uses the rule established in the joint analysis: Kneedle elbow on the
   best-of-seeds error curve, restricted to $K$ where **every seed converged**.

4. **Matching is done on line-centred profiles.** Each line's archetypes are centred on that line's
   own cell centroid before comparison. This is the post-hoc analogue of the offset correction: it
   asks "is this the same *state relative to its line*", not "do these have the same absolute
   staining", which would just re-discover that MCF7 is ER-high.

5. **Similarity is calibrated against a null.** Cosine similarity between two archetype vectors is
   meaningless without knowing what similarity arises by chance, so every match is scored against a
   permutation null.

## 0. Setup

In [ ]:
import sys, json, itertools
from pathlib import Path

CT_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Code/cytof-transform"
CE_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Experiments/CyEmbed"
for p in (CT_PATH, CE_PATH):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

from cytofstandard import Project
from CyEmbed.data import extract_matrix, fit_scaler, preprocess_array, split_train_val_indices
from CyEmbed.train import build_sweep_configs, run_sweep
from CyEmbed.analysis import load_run_outputs

plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 10, "ytick.labelsize": 10})

BASE = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
PLOTS = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)
PERLINE_DIR = BASE / "outputs/cyembed_perline_sweep"
JOINT_DIR = BASE / "outputs/cyembed_joint_sweep"

LINES = ["MDAMB468", "HCC70", "SUM149", "HCC1937", "MCF7"]
LINE_DISP = {"MDAMB468": "MDA-MB-468", "HCC70": "HCC70", "SUM149": "SUM149",
             "HCC1937": "HCC1937", "MCF7": "MCF7"}
DISP_ORDER = ["HCC1937", "HCC70", "MCF7", "MDA-MB-468", "SUM149"]
LINE_COLORS = dict(zip(DISP_ORDER, ["#66c2a5", "#8da0cb", "#a6d854", "#fc8d62", "#b3b3b3"]))

SEED = 42
K_RANGE = list(range(2, 13))
SEEDS = [42, 1, 2]
FAILURE_TOLERANCE = 0.05      # >5% above the best seed at a given K = optimiser failure

BASE_CONFIG = dict(
    model_type="deterministic", decoder_type="factorized",
    d=16, hidden_dims=[64, 32], tau=1.0,
    epochs=1500, early_stopping=True, patience=20,
    min_delta=0.0, restore_best_weights=True,
    lr=1e-3, batch_size=2048, weight_decay=1e-5, dropout=0.0,
    logit_normalizer="entmax", entmax_alpha=1.5, grad_clip_norm=5.0,
    separation_mode="cosine_sq", balance_mode="l2_uniform",
    lambda_entropy=1e-3, lambda_sep=1e-3, lambda_balance=0.05,
    recon_loss_type="mse", device="cpu", deterministic=True, seed=SEED,
)
print("setup complete")

## 1. Data and the shared scaler

All five lines are loaded and the scaler is fitted once across them, exactly as in
`scripts/cyembed_perline_sweep.py`. Each line's matrix is then transformed with that shared scaler.

In [ ]:
adatas = {}
for line in LINES:
    proj = Project.load(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{line}_NormCompare")
    adata = proj.get_run(line).read_adata()
    bio = [m for m in adata.var_names if m not in ["H3", "H3.3", "H4"]]
    sub = ad.AnnData(X=adata[:, bio].layers["norm_divide"].copy(),
                     obs=adata.obs[["cell_uuid", "line_id"]].copy(),
                     var=adata[:, bio].var.copy())
    sub.obs["cell_line"] = LINE_DISP[line]
    adatas[line] = sub

combined = ad.concat([adatas[l] for l in LINES], join="outer")
bundle_all = extract_matrix(adata=combined, layer=None, sample_col="cell_line")
SHARED_SCALER, _ = fit_scaler(bundle_all.X, mode="zscore",
                              sample_ids=bundle_all.sample_ids, balanced_max_per_sample=5000)
MARKERS = list(bundle_all.marker_names)

line_data = {}
for line in LINES:
    b = extract_matrix(adata=adatas[line], layer=None, sample_col="cell_line")
    x = preprocess_array(b.X, SHARED_SCALER)
    tr, va = split_train_val_indices(n_cells=len(x), val_fraction=0.2, seed=SEED, stratify_labels=None)
    line_data[line] = dict(X=x, bundle=b, train_idx=tr, val_idx=va)

print(f"{len(MARKERS)} markers | shared scaler fitted on {len(bundle_all.X):,} cells\n")
for line in LINES:
    x = line_data[line]["X"]
    print(f"  {LINE_DISP[line]:<11} {x.shape[0]:>7,} cells   "
          f"train {len(line_data[line]['train_idx']):>6,} / val {len(line_data[line]['val_idx']):>6,}")

## 2. Per-line sweeps

`K = 2..12` × seeds {42, 1, 2}, independently per line. Completed runs are reused by fingerprint, so
re-running this cell is cheap. The same grid can be run outside the notebook with
`scripts/cyembed_perline_sweep.py --line all`.

In [ ]:
configs = build_sweep_configs({"K": K_RANGE, "seed": SEEDS})
print(f"{len(configs)} configurations per line x {len(LINES)} lines = {len(configs)*len(LINES)} runs\n")

for line in LINES:
    out_dir = PERLINE_DIR / line
    if out_dir.exists():
        stale = [d for d in out_dir.glob("run_*") if d.is_dir() and not any(d.iterdir())]
        for d in stale:
            d.rmdir()
        if stale:
            print(f"[{line}] cleared {len(stale)} empty run dir(s)")
    ld = line_data[line]
    run_sweep(x=ld["X"], marker_names=list(ld["bundle"].marker_names),
              cell_ids=list(ld["bundle"].cell_ids), output_root=out_dir,
              base_config=BASE_CONFIG, sweep_configs=configs,
              train_idx=ld["train_idx"], val_idx=ld["val_idx"],
              sample_ids=None, scaler_state=SHARED_SCALER.to_dict())
print("\nall per-line sweeps present")

In [ ]:
# Inventory every per-line run
rows = []
for line in LINES:
    for r in sorted((PERLINE_DIR / line).glob("run_*")):
        sm_f, cf_f = r / "summary_metrics.json", r / "config.json"
        if not (sm_f.exists() and cf_f.exists()):
            continue
        sm, cfg = json.loads(sm_f.read_text()), json.loads(cf_f.read_text())
        rows.append({"line": LINE_DISP[line], "line_key": line, "K": cfg["K"], "seed": cfg["seed"],
                     "val_recon": sm.get("best_val_recon", sm.get("val", {}).get("recon_mse")),
                     "run_dir": str(r)})
runs_df = pd.DataFrame(rows)

# Completeness check -- a missing cell would bias the per-line curves invisibly.
want = {(LINE_DISP[l], k, s) for l in LINES for k in K_RANGE for s in SEEDS}
have = {(r.line, r.K, r.seed) for r in runs_df.itertuples()}
missing = sorted(want - have)
print(f"runs found: {len(runs_df)} / {len(want)}")
if missing:
    print(f"!! MISSING {len(missing)}: {missing[:10]}{' ...' if len(missing) > 10 else ''}")
else:
    print("grid complete for every line")
print()
print(runs_df.groupby(["line", "K"]).size().unstack(fill_value=0).to_string())

## 3. Choosing $K$ for each line

Same rule as the joint analysis, applied independently per line:

1. A seed scoring more than 5% above the best seed at the same $K$ is an **optimiser failure**, not
   evidence about $K$. Failures are reported and excluded from the error curve.
2. **Kneedle elbow** on the best-of-seeds curve.
3. **Selected $K$** = largest $K$ at or below the elbow at which every seed converged.

Cross-seed stability (Hungarian-matched cosine of $\hat{A}$ rows) is reported as a diagnostic — it
also tells us how much of the later cross-line matching signal we can trust, since an archetype set
that is not reproducible across seeds within a line cannot be reliably matched across lines.

In [ ]:
def kneedle_elbow(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 3:
        return x[0] if len(x) else None
    xn = (x - x.min()) / (np.ptp(x) + 1e-12)
    yn = (y - y.min()) / (np.ptp(y) + 1e-12)
    chord = yn[0] + (xn - xn[0]) * (yn[-1] - yn[0]) / (xn[-1] - xn[0] + 1e-12)
    return x[int(np.argmax(chord - yn))]


def matched_cosine(a1, a2):
    """Mean cosine after optimal one-to-one archetype matching (permutation-invariant)."""
    n1 = a1 / (np.linalg.norm(a1, axis=1, keepdims=True) + 1e-12)
    n2 = a2 / (np.linalg.norm(a2, axis=1, keepdims=True) + 1e-12)
    sim = n1 @ n2.T
    r, c = linear_sum_assignment(-sim)
    return float(sim[r, c].mean())


def load_A(run_dir):
    return np.load(Path(run_dir) / "A_hat.npy")


sel_rows, failures = [], []
for line_disp, g_line in runs_df.groupby("line"):
    g_line = g_line.copy()
    g_line["best_at_K"] = g_line.groupby("K")["val_recon"].transform("min")
    g_line["excess"] = g_line["val_recon"] / g_line["best_at_K"] - 1.0
    g_line["converged"] = g_line["excess"] <= FAILURE_TOLERANCE
    failures.append(g_line[~g_line["converged"]])

    per_K = []
    for k, g in g_line.groupby("K"):
        A = {int(r.seed): load_A(r.run_dir) for r in g.itertuples()}
        sd = sorted(A)
        sims = [matched_cosine(A[i], A[j]) for ii, i in enumerate(sd) for j in sd[ii + 1:]]
        per_K.append({"line": line_disp, "K": k, "n_seeds": len(g),
                      "n_converged": int(g["converged"].sum()),
                      "val_best": g["val_recon"].min(),
                      "val_mean_conv": g.loc[g["converged"], "val_recon"].mean(),
                      "stability": float(np.mean(sims)) if sims else np.nan,
                      "_A": A, "_g": g})
    per_K = pd.DataFrame(per_K).sort_values("K").reset_index(drop=True)
    elbow = kneedle_elbow(per_K["K"].values, per_K["val_best"].values)
    ok = per_K[(per_K["K"] <= elbow) & (per_K["n_converged"] == per_K["n_seeds"])]
    K_sel = int(ok["K"].max()) if len(ok) else int(per_K.loc[per_K["val_best"].idxmin(), "K"])
    per_K["elbow"] = elbow; per_K["K_sel"] = K_sel
    sel_rows.append(per_K)

sel_all = pd.concat(sel_rows, ignore_index=True)
failures = pd.concat(failures, ignore_index=True)

print("=== Convergence failures (>5% above best seed at that K) ===")
if len(failures) == 0:
    print("  none -- every per-line run converged comparably across seeds")
else:
    print(f"  {len(failures)} of {len(runs_df)} runs:")
    for _, r in failures.sort_values(["line", "K", "seed"]).iterrows():
        print(f"    {r['line']:<11} K={int(r['K']):<3} seed={int(r['seed']):<3} "
              f"{r['val_recon']:.5f} vs best {r['best_at_K']:.5f}  (+{100*r['excess']:.1f}%)")

print("\n=== Per-line K selection ===")
summary = (sel_all.groupby("line")
           .agg(elbow=("elbow", "first"), K_selected=("K_sel", "first")).reset_index())
summary["val_at_K"] = [sel_all[(sel_all.line == r.line) & (sel_all.K == r.K_selected)]["val_best"].iloc[0]
                       for r in summary.itertuples()]
summary["stability_at_K"] = [sel_all[(sel_all.line == r.line) & (sel_all.K == r.K_selected)]["stability"].iloc[0]
                             for r in summary.itertuples()]
summary["n_cells"] = [line_data[[k for k, v in LINE_DISP.items() if v == r.line][0]]["X"].shape[0]
                      for r in summary.itertuples()]
print(summary.round(4).to_string(index=False))

K_SEL = dict(zip(summary["line"], summary["K_selected"]))
print(f"\nSelected K per line: {K_SEL}")
print(f"Total archetypes across all lines: {sum(K_SEL.values())}")

In [ ]:
# Per-line selection diagnostics
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for line in DISP_ORDER:
    s = sel_all[sel_all.line == line].sort_values("K")
    c = LINE_COLORS[line]
    axes[0].plot(s["K"], s["val_best"], marker="o", color=c, lw=1.8, label=line)
    axes[1].plot(s["K"], s["stability"], marker="s", color=c, lw=1.8, label=line)
    ks = K_SEL[line]
    axes[0].scatter([ks], s.loc[s.K == ks, "val_best"], s=190, facecolors="none",
                    edgecolors=c, linewidths=2.5, zorder=5)
    axes[1].scatter([ks], s.loc[s.K == ks, "stability"], s=190, facecolors="none",
                    edgecolors=c, linewidths=2.5, zorder=5)

axes[0].set_xlabel("K"); axes[0].set_ylabel("Validation recon loss (best of seeds)")
axes[0].set_title("Per-line reconstruction vs K\n(circled = selected K)", fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.5); axes[0].legend(fontsize=9)

axes[1].set_xlabel("K"); axes[1].set_ylabel("Mean matched cosine across seeds")
axes[1].set_title("Per-line cross-seed stability\n(circled = selected K)", fontweight="bold")
axes[1].set_ylim(0, 1.02); axes[1].grid(True, linestyle="--", alpha=0.5); axes[1].legend(fontsize=9)

bars = axes[2].bar([l for l in DISP_ORDER], [K_SEL[l] for l in DISP_ORDER],
                   color=[LINE_COLORS[l] for l in DISP_ORDER], edgecolor="black", linewidth=0.6)
for b, l in zip(bars, DISP_ORDER):
    axes[2].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.08, str(K_SEL[l]),
                 ha="center", fontweight="bold")
axes[2].set_ylabel("Selected K"); axes[2].set_title("Archetypes needed per line", fontweight="bold")
axes[2].tick_params(axis="x", rotation=20)
axes[2].grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_PerLine_K_Selection.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Extracting a representative model per line

For each line we take the model at its selected $K$, using the **medoid seed** — the seed whose
$\hat{A}$ is most similar on average to the other seeds — rather than the lowest-error seed, which
would cherry-pick initialisation noise.

Two representations are kept per archetype:

- **`A_raw`** — the fitted vertex in the shared z-space.
- **`A_cen`** — the same vertex minus that line's own cell centroid, i.e. the *direction from the
  line's centre toward the archetype*. This is what matching uses. Without centring, every
  comparison would be dominated by the fact that MCF7 sits at a different place in marker space than
  the basal lines, and we would simply rediscover line identity instead of state correspondence.

In [ ]:
per_line_model = {}
for line in DISP_ORDER:
    s = sel_all[(sel_all.line == line) & (sel_all.K == K_SEL[line])].iloc[0]
    A_by_seed, g = s["_A"], s["_g"]
    conv = set(g.loc[g["converged"], "seed"].astype(int)) if "converged" in g else set(A_by_seed)
    cand = [sd for sd in sorted(A_by_seed) if sd in conv] or sorted(A_by_seed)
    if len(cand) > 1:
        mean_sim = {sd: np.mean([matched_cosine(A_by_seed[sd], A_by_seed[o])
                                 for o in cand if o != sd]) for sd in cand}
        medoid = max(mean_sim, key=mean_sim.get)
    else:
        medoid, mean_sim = cand[0], {cand[0]: np.nan}

    run_dir = g.loc[g["seed"] == medoid, "run_dir"].iloc[0]
    out = load_run_outputs(run_dir)
    line_key = [k for k, v in LINE_DISP.items() if v == line][0]
    X_line = line_data[line_key]["X"]
    centroid = X_line.mean(axis=0)

    per_line_model[line] = {
        "line_key": line_key, "K": int(K_SEL[line]), "seed": int(medoid),
        "run_dir": run_dir, "A_raw": out["A_hat"], "A_cen": out["A_hat"] - centroid,
        "W": out["W"], "centroid": centroid, "medoid_sim": mean_sim.get(medoid, np.nan),
        "markers": out["marker_names"],
    }
    print(f"{line:<11} K={K_SEL[line]:<3} medoid seed={medoid:<3} "
          f"(mean matched cosine to other seeds = {mean_sim.get(medoid, float('nan')):.3f})")

assert all(per_line_model[l]["markers"] == MARKERS for l in DISP_ORDER), "marker order mismatch"
TOTAL_ARCH = sum(m["K"] for m in per_line_model.values())
print(f"\n{TOTAL_ARCH} archetypes total across {len(DISP_ORDER)} lines")

## 5. Cross-line similarity, calibrated against a null

Archetypes are compared by **cosine similarity of their line-centred profiles**. Cosine on its own is
not interpretable — a value of 0.5 could be strong or meaningless depending on the dimensionality and
the marker correlation structure. So similarity is calibrated two ways:

- **Marker-permutation null.** Shuffling the marker order within one archetype destroys any real
  correspondence while preserving its magnitude distribution exactly. The distribution of similarities
  under this shuffle gives the chance level.
- **Match threshold** = the 99th percentile of that null. Pairs above it are treated as genuine
  correspondences.

In [ ]:
def unit(v):
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)


# Stack every archetype from every line into one table.
rows, vecs_cen, vecs_raw = [], [], []
for line in DISP_ORDER:
    m = per_line_model[line]
    for k in range(m["K"]):
        rows.append({"label": f"{line}|A{k+1}", "line": line, "k": k + 1,
                     "pool_pct": 100 * float((m["W"].argmax(1) == k).mean())})
        vecs_cen.append(m["A_cen"][k]); vecs_raw.append(m["A_raw"][k])

arch_df = pd.DataFrame(rows)
A_CEN = np.vstack(vecs_cen); A_RAW = np.vstack(vecs_raw)
U = unit(A_CEN)
S = U @ U.T                                        # cosine similarity, line-centred
same_line = arch_df["line"].values[:, None] == arch_df["line"].values[None, :]

# Two nulls, because a cosine value is not interpretable on its own.
#  (a) marker-permutation: shuffles marker order within one vector, destroying correspondence while
#      preserving that vector's magnitude distribution exactly. Structure-preserving and stringent.
#  (b) random-direction: isotropic Gaussian vectors. The naive baseline; much weaker.
rng = np.random.default_rng(SEED)
null_perm, null_rand = [], []
for _ in range(4000):
    i, j = rng.integers(0, len(U), 2)
    if arch_df["line"].iloc[i] == arch_df["line"].iloc[j]:
        continue
    null_perm.append(float(unit(A_CEN[i][rng.permutation(A_CEN.shape[1])]) @ U[j]))
    null_rand.append(float(unit(rng.normal(size=A_CEN.shape[1])) @ U[j]))
null_perm, null_rand = np.array(null_perm), np.array(null_rand)

MATCH_THRESHOLD = float(np.quantile(np.abs(null_perm), 0.99))
LENIENT_THRESHOLD = float(np.quantile(np.abs(null_rand), 0.99))

off = S[~same_line]
print(f"Null (a) marker-permuted, n={len(null_perm)}: mean |cos| = {np.abs(null_perm).mean():.3f}, "
      f"99th pct = {MATCH_THRESHOLD:.3f}   <-- used as the match threshold")
print(f"Null (b) random direction, n={len(null_rand)}: mean |cos| = {np.abs(null_rand).mean():.3f}, "
      f"99th pct = {LENIENT_THRESHOLD:.3f}")
print(f"\nObserved cross-line cosine: mean = {off.mean():.3f}, max = {off.max():.3f}")
print(f"  exceeding the STRINGENT threshold ({MATCH_THRESHOLD:.3f}): {100*(off > MATCH_THRESHOLD).mean():.1f}% of pairs")
print(f"  exceeding the lenient  threshold ({LENIENT_THRESHOLD:.3f}): {100*(off > LENIENT_THRESHOLD).mean():.1f}% of pairs")
print("\nThe permutation null is deliberately conservative: archetype vectors are dominated by a few")
print("large marker loadings, so a shuffled copy can still align by chance. Matches surviving it are")
print("strong claims; the gap between the two thresholds is the grey zone.")

In [ ]:
# Full cross-line similarity heatmap
order = arch_df.index.tolist()
fig, ax = plt.subplots(figsize=(0.42 * len(order) + 4, 0.42 * len(order) + 3))
Sm = S.copy(); np.fill_diagonal(Sm, np.nan)
sns.heatmap(pd.DataFrame(Sm, index=arch_df["label"], columns=arch_df["label"]),
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "cosine similarity (line-centred)", "shrink": 0.6}, ax=ax)

# Mark line blocks
bounds = np.cumsum([per_line_model[l]["K"] for l in DISP_ORDER])
for b in bounds[:-1]:
    ax.axhline(b, color="black", lw=1.6); ax.axvline(b, color="black", lw=1.6)
ax.set_title("Cross-line archetype similarity\n"
             f"(black blocks = same line; matches are off-block pairs above {MATCH_THRESHOLD:.2f})",
             fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_PerLine_Similarity_Matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Pairwise line-to-line matching

For each pair of lines, archetypes are matched **one-to-one** by maximising total cosine similarity
(Hungarian assignment). Because lines may have different $K$, the smaller line's archetypes are all
matched and the surplus on the larger side is left unmatched — which is itself informative: a
consistently unmatched archetype is a candidate line-specific state.

In [ ]:
pair_records, pair_scores = [], pd.DataFrame(index=DISP_ORDER, columns=DISP_ORDER, dtype=float)
for l1, l2 in itertools.combinations(DISP_ORDER, 2):
    i1 = arch_df.index[arch_df.line == l1].values
    i2 = arch_df.index[arch_df.line == l2].values
    sub = S[np.ix_(i1, i2)]
    r, c = linear_sum_assignment(-sub)
    sims = sub[r, c]
    pair_scores.loc[l1, l2] = pair_scores.loc[l2, l1] = sims.mean()
    for a, b, v in zip(r, c, sims):
        pair_records.append({"line_a": l1, "arch_a": arch_df.label[i1[a]],
                             "line_b": l2, "arch_b": arch_df.label[i2[b]],
                             "cosine": v, "matched": v > MATCH_THRESHOLD})
pairs_df = pd.DataFrame(pair_records)
np.fill_diagonal(pair_scores.values, 1.0)

print("=== Mean matched cosine between line pairs (Hungarian 1-1) ===")
print(pair_scores.astype(float).round(3).to_string())
print(f"\nOf {len(pairs_df)} optimal pairings, {int(pairs_df['matched'].sum())} "
      f"({100*pairs_df['matched'].mean():.0f}%) exceed the null threshold {MATCH_THRESHOLD:.3f}")

print("\n=== Strongest cross-line correspondences ===")
print(pairs_df.nlargest(12, "cosine")[["arch_a", "arch_b", "cosine"]].round(3).to_string(index=False))
print("\n=== Weakest optimal pairings (candidate line-specific states) ===")
print(pairs_df.nsmallest(8, "cosine")[["arch_a", "arch_b", "cosine"]].round(3).to_string(index=False))

## 7. Meta-archetypes: which states are conserved across lines?

Pairwise matching answers "does line A have a counterpart in line B", but with five lines and
different $K$ per line the useful question is global: **how many distinct states exist across the
panel, and how many lines does each appear in?**

All archetypes from all lines are pooled and clustered by cosine distance (average linkage). The
dendrogram is cut at the **null-calibrated match threshold** from §5, so two archetypes join a cluster
only when they are more similar than chance. Each resulting cluster is a **meta-archetype**, labelled
by how many distinct lines contribute to it:

- **Conserved** — present in all 5 lines
- **Shared** — present in 3–4 lines
- **Restricted** — present in 2 lines
- **Line-specific** — present in 1 line only

A cluster containing two archetypes from the *same* line is flagged, since that indicates the cut is
merging states one line resolves separately.

In [ ]:
D = 1.0 - S
np.fill_diagonal(D, 0.0)
D = np.clip((D + D.T) / 2, 0, None)
Z = linkage(squareform(D, checks=False), method="average")

cut_distance = 1.0 - MATCH_THRESHOLD
cluster_id = fcluster(Z, t=cut_distance, criterion="distance")
arch_df["meta"] = cluster_id

meta_rows = []
for m, g in arch_df.groupby("meta"):
    lines_in = sorted(set(g["line"]))
    n_lines = len(lines_in)
    kind = ({5: "CONSERVED (all 5 lines)", 4: "shared (4 lines)", 3: "shared (3 lines)",
             2: "restricted (2 lines)", 1: "LINE-SPECIFIC"}[n_lines])
    idx = g.index.values
    within = S[np.ix_(idx, idx)][np.triu_indices(len(idx), 1)]
    meta_rows.append({
        "meta": m, "n_archetypes": len(g), "n_lines": n_lines, "kind": kind,
        "lines": ", ".join(lines_in),
        "mean_within_cos": float(within.mean()) if len(within) else np.nan,
        "mean_pool_pct": float(g["pool_pct"].mean()),
        "duplicate_line": bool(g["line"].duplicated().any()),
        "members": ", ".join(g["label"]),
    })
meta_df = pd.DataFrame(meta_rows).sort_values(["n_lines", "mean_pool_pct"],
                                              ascending=[False, False]).reset_index(drop=True)

print(f"Cut at cosine >= {MATCH_THRESHOLD:.3f} (distance {cut_distance:.3f}) -> "
      f"{len(meta_df)} meta-archetypes from {TOTAL_ARCH} per-line archetypes\n")
print(meta_df[["meta", "n_lines", "kind", "n_archetypes", "mean_within_cos",
               "mean_pool_pct", "lines"]].round(3).to_string(index=False))

print("\n=== Breakdown ===")
for kind, g in meta_df.groupby("kind", sort=False):
    print(f"  {kind:<24} {len(g)} meta-archetype(s), "
          f"{int(g['n_archetypes'].sum())} per-line archetypes")
dupes = meta_df[meta_df["duplicate_line"]]
if len(dupes):
    print(f"\n  NOTE: {len(dupes)} cluster(s) contain >1 archetype from the same line, i.e. the cut "
          "merges states that\n  a line resolves separately:")
    for _, r in dupes.iterrows():
        print(f"    meta {r['meta']}: {r['members']}")

In [ ]:
# Membership map: meta-archetype x line
present = pd.crosstab(arch_df["meta"], arch_df["line"]).reindex(columns=DISP_ORDER, fill_value=0)
present = present.reindex(meta_df["meta"].values)
labels = [f"M{r.meta} ({r.n_lines}/5)" for r in meta_df.itertuples()]

fig, axes = plt.subplots(1, 2, figsize=(17, 0.42 * len(meta_df) + 4),
                         gridspec_kw={"width_ratios": [1.15, 1]})

sns.heatmap(present.values, annot=True, fmt="d", cmap="Blues", vmin=0, vmax=2,
            xticklabels=DISP_ORDER, yticklabels=labels, linewidths=0.6, linecolor="white",
            cbar_kws={"label": "archetypes contributed"}, ax=axes[0])
axes[0].set_title("Meta-archetype membership by cell line\n"
                  "(rows sorted by conservation, then abundance)", fontweight="bold")
axes[0].set_xlabel("Cell line"); axes[0].set_ylabel("Meta-archetype")
axes[0].tick_params(axis="x", rotation=25)

colors = {"CONSERVED (all 5 lines)": "#1a9850", "shared (4 lines)": "#91cf60",
          "shared (3 lines)": "#d9ef8b", "restricted (2 lines)": "#fee08b",
          "LINE-SPECIFIC": "#d73027"}
axes[1].barh(range(len(meta_df)), meta_df["mean_pool_pct"],
             color=[colors[k] for k in meta_df["kind"]], edgecolor="black", linewidth=0.5)
axes[1].set_yticks(range(len(meta_df))); axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].set_xlabel("Mean % of cells dominated (within contributing lines)")
axes[1].set_title("How much of each line these states account for", fontweight="bold")
axes[1].grid(True, axis="x", linestyle="--", alpha=0.4)
handles = [plt.Rectangle((0, 0), 1, 1, fc=c, ec="black") for c in colors.values()]
axes[1].legend(handles, colors.keys(), fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_PerLine_MetaArchetypes.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# What is each meta-archetype, biologically? Use the mean line-centred profile of its members.
meta_profiles = {}
print("=== Meta-archetype marker signatures (mean line-centred profile) ===")
for r in meta_df.itertuples():
    idx = arch_df.index[arch_df.meta == r.meta].values
    prof = pd.Series(A_CEN[idx].mean(0), index=MARKERS)
    meta_profiles[r.meta] = prof
    hi = ", ".join(f"{m}{prof[m]:+.1f}" for m in prof.sort_values(ascending=False).index[:6])
    lo = ", ".join(f"{m}{prof[m]:+.1f}" for m in prof.sort_values().index[:6])
    print(f"\nM{r.meta}  [{r.kind}]  members: {r.members}")
    print(f"   HIGH: {hi}")
    print(f"   LOW : {lo}")

meta_prof_df = pd.DataFrame(meta_profiles).T
meta_prof_df.index = [f"M{m}" for m in meta_prof_df.index]

fig, ax = plt.subplots(figsize=(15, 0.45 * len(meta_prof_df) + 3))
v = float(np.abs(meta_prof_df.values).max())
sns.heatmap(meta_prof_df, cmap="RdBu_r", center=0, vmin=-v, vmax=v,
            cbar_kws={"label": "line-centred profile (z units)"}, ax=ax)
ax.set_title("Meta-archetype marker signatures\n(mean of member archetypes, centred on each line)",
             fontweight="bold", pad=12)
ax.set_yticklabels([f"{lab}  ({r.n_lines}/5 lines)" for lab, r in zip(meta_prof_df.index,
                                                                     meta_df.itertuples())],
                   rotation=0)
plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_PerLine_MetaArchetype_Profiles.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Does the global chromatin axis reappear independently in every line?

The joint analysis found that two of its nine archetypes were the low and high poles of a single
**global chromatin-magnitude axis** (all histone PTMs moving together, correlating +0.75 with KI67)
rather than two distinct states. A fair objection is that this could be an artifact of the joint
model — a consequence of pooling five lines.

Fitting each line separately is a clean test. If every line independently spends vertices on the same
global axis, it is real structure in the data. If it only appears when lines are pooled, it was an
artifact of the joint fit.

In [ ]:
HIST = [m for m in MARKERS if m.startswith(("H2A", "H3", "H4")) and m != "H3S28p"]
hist_idx = [MARKERS.index(m) for m in HIST]
u_global = np.zeros(len(MARKERS)); u_global[hist_idx] = 1.0
u_global /= np.linalg.norm(u_global)

print(f"Global chromatin direction = mean of {len(HIST)} histone-PTM channels\n")
rows = []
for line in DISP_ORDER:
    m = per_line_model[line]
    lk = m["line_key"]; X = line_data[lk]["X"]
    gh = X[:, hist_idx].mean(1)
    ki = X[:, MARKERS.index("KI67")]
    # variance share of the global direction within this line
    share = 100 * (X @ u_global).var() / X.var(axis=0).sum()
    rs = np.array([np.corrcoef(m["W"][:, k], gh)[0, 1] for k in range(m["K"])])
    n_axis = int((np.abs(rs) > 0.45).sum())
    dom = m["W"].argmax(1)
    cov = 100 * np.isin(dom, np.flatnonzero(np.abs(rs) > 0.45)).mean()
    rows.append({"line": line, "K": m["K"], "global_var_%": share,
                 "r_gh_KI67": np.corrcoef(gh, ki)[0, 1],
                 "vertices_on_axis": n_axis, "max_|r|": np.abs(rs).max(),
                 "pct_cells_on_axis": cov,
                 "both_poles": bool((rs > 0.45).any() and (rs < -0.45).any())})
axis_df = pd.DataFrame(rows)
print("=== Global chromatin axis, fitted independently per line ===")
print(axis_df.round(3).to_string(index=False))

n_with = int((axis_df["vertices_on_axis"] > 0).sum())
print(f"\n-> {n_with}/{len(DISP_ORDER)} lines independently devote at least one vertex to this axis "
      f"(mean {axis_df['pct_cells_on_axis'].mean():.0f}% of their cells).")
print(f"-> Within-line correlation of the axis with KI67: "
      f"{axis_df['r_gh_KI67'].min():.2f} to {axis_df['r_gh_KI67'].max():.2f}")
print("   The axis is therefore a property of the data, not of pooling the lines." if n_with >= 4
      else "   The axis is NOT consistently reproduced per line -- it may be a joint-model artifact.")

## 9. Do the per-line results agree with the joint model?

Two independent routes to the same question — a shared basis fitted jointly with offset correction,
versus five independent bases matched after the fact. Where they agree, the conclusion is robust to
the modelling strategy; where they disagree, the joint model's shared-basis assumption is doing work.

Each joint archetype is matched to its closest meta-archetype (line-centred, cosine).

In [ ]:
joint_run = JOINT_DIR / "run_deterministic_37e350d66f"
joint = load_run_outputs(joint_run)
assert joint["marker_names"] == MARKERS, "marker order mismatch vs joint model"
A_joint = joint["A_hat"]
joint_centroid = joint["X"].mean(axis=0)
A_joint_cen = A_joint - joint_centroid
Uj = unit(A_joint_cen)
Um = unit(meta_prof_df.values)

Sjm = Uj @ Um.T
jm = pd.DataFrame(Sjm, index=[f"Joint A{k+1}" for k in range(A_joint.shape[0])],
                  columns=meta_prof_df.index)

best = jm.idxmax(axis=1); bestv = jm.max(axis=1)
nlines = dict(zip([f"M{m}" for m in meta_df["meta"]], meta_df["n_lines"]))
cmp_tbl = pd.DataFrame({"closest_meta": best, "cosine": bestv.round(3),
                        "meta_n_lines": [nlines[b] for b in best],
                        "above_null": bestv > MATCH_THRESHOLD})
print("=== Joint archetypes vs per-line meta-archetypes ===")
print(cmp_tbl.to_string())
print(f"\n{int(cmp_tbl['above_null'].sum())}/{len(cmp_tbl)} joint archetypes have a per-line "
      f"counterpart above the null threshold ({MATCH_THRESHOLD:.3f})")

unmatched_meta = [c for c in jm.columns if c not in set(best)]
print(f"\nMeta-archetypes with no joint counterpart: "
      f"{unmatched_meta if unmatched_meta else 'none'}")

fig, ax = plt.subplots(figsize=(0.55 * len(jm.columns) + 5, 0.5 * len(jm) + 3))
sns.heatmap(jm, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f",
            annot_kws={"size": 7}, cbar_kws={"label": "cosine (line-centred)"}, ax=ax)
ax.set_title("Joint model (K=9) vs independently-fitted per-line meta-archetypes\n"
             "Agreement here means the shared-basis assumption was not driving the result",
             fontweight="bold", pad=12)
ax.set_xlabel("Per-line meta-archetype"); ax.set_ylabel("Joint archetype")
plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_PerLine_vs_Joint.png", dpi=200, bbox_inches="tight")
plt.show()

## 9b. Hunting the minor basal-like population in MCF7

MCF7 is luminal, but a small CD44⁺/CD24⁻ stem-like subpopulation is repeatedly reported in it. A
first pass at the K=6 fit did **not** find one, and that conclusion was wrong for two avoidable
reasons:

1. **The basal score averaged KRT5 with CD49f.** MCF7's basal-like cells gain CD44 and CD49f but stay
   comparatively KRT5-low, so averaging the two markers diluted the signal below detection.
2. **K=6 is far too coarse for a ~1% population.** At K=6 these cells are smeared into a 6.9%
   archetype at only 3.1× enrichment. The state only separates at K≥10.

This section searches for the population directly rather than relying on the selected-K archetypes,
and validates it three ways: reproducibility across independent fits, absolute comparison against the
basal lines, and where the cells land in the joint model.

In [ ]:
# --- 9b.1 Direct search across higher-K MCF7 fits -------------------------------------------------
mcf7_cfg = {}
for f in (PERLINE_DIR / "MCF7").glob("run_*/config.json"):
    c = json.loads(f.read_text())
    mcf7_cfg[(c["K"], c["seed"])] = f.parent

X_mcf7 = line_data["MCF7"]["X"]
S = pd.DataFrame(X_mcf7, columns=MARKERS)
gate_cd44 = ((S["CD44"] > 0) & (S["CD24"] < 0)).values
BASAL_MK, LUM_MK = ["CD44", "CD49f", "KRT5"], ["CD24", "GATA3", "ER"]

print(f"MCF7: {len(S):,} cells | CD44+/CD24- gate = {gate_cd44.sum():,} ({100*gate_cd44.mean():.2f}%)\n")

def most_basal_archetype(K, seed, min_cells=300):
    """Archetype with the highest (basal - luminal) mean profile among its dominated cells."""
    W = np.load(mcf7_cfg[(K, seed)] / "W.npy")
    dom = W.argmax(1)
    best = None
    for k in range(K):
        m = dom == k
        if m.sum() < min_cells:
            continue
        sc = S.loc[m, BASAL_MK].mean().mean() - S.loc[m, LUM_MK].mean().mean()
        if best is None or sc > best[1]:
            best = (k, sc, m)
    return best

print("=== Most basal-leaning archetype in each independent MCF7 fit ===")
sets = {}
for K in [10, 11, 12]:
    for seed in SEEDS:
        k, sc, m = most_basal_archetype(K, seed)
        sets[(K, seed)] = m
        print(f"  K={K} seed={seed:<3} A{k+1}: n={m.sum():>5,} ({100*m.mean():4.1f}%)  "
              f"basal-luminal {sc:+.2f}  CD44+/CD24- enrichment "
              f"{100*gate_cd44[m].mean()/(100*gate_cd44.mean()):.1f}x")

# Consensus: cells selected by a majority of the 9 independent fits.
votes = np.sum([sets[k].astype(int) for k in sets], axis=0)
MCF7_BASAL = votes >= 5
keys = list(sets)
jac = np.array([[(sets[a] & sets[b]).sum() / max((sets[a] | sets[b]).sum(), 1)
                 for b in keys] for a in keys])
print(f"\nCell-membership agreement across fits: mean Jaccard = "
      f"{jac[~np.eye(len(keys), dtype=bool)].mean():.2f}")
print(f"CONSENSUS population (>=5 of 9 fits): {MCF7_BASAL.sum():,} cells "
      f"({100*MCF7_BASAL.mean():.2f}% of MCF7)")

In [ ]:
# --- 9b.2 Phenotype, in absolute terms against the basal lines -----------------------------------
diff = (S[MCF7_BASAL].mean() - S[~MCF7_BASAL].mean()).sort_values()
print("=== MCF7 basal-like consensus vs the rest of MCF7 ===")
print("  UP  :", ", ".join(f"{a}{b:+.2f}" for a, b in diff[::-1][:8].items()))
print("  DOWN:", ", ".join(f"{a}{b:+.2f}" for a, b in diff[:8].items()))

ref_means = {}
for line in DISP_ORDER:
    if line == "MCF7":
        continue
    lk = per_line_model[line]["line_key"]
    ref_means[line] = pd.Series(line_data[lk]["X"].mean(0), index=MARKERS)

print("\n=== Absolute values: is this population basal by the standards of the basal lines? ===")
rows = []
for mk in ["CD44", "CD49f", "KRT5", "CD24", "GATA3", "ER", "KRT8-18", "Vimentin", "KI67"]:
    r = {"marker": mk, "MCF7_basal_like": S.loc[MCF7_BASAL, mk].mean(),
         "MCF7_bulk": S[mk].mean()}
    r.update({l: ref_means[l][mk] for l in ref_means})
    rows.append(r)
absdf = pd.DataFrame(rows).set_index("marker")
print(absdf.round(2).to_string())

in_range = []
for mk in ["CD44", "CD49f", "KRT5"]:
    lo = min(ref_means[l][mk] for l in ref_means)
    hi = max(ref_means[l][mk] for l in ref_means)
    v = S.loc[MCF7_BASAL, mk].mean()
    in_range.append((mk, v, lo, hi, lo <= v <= hi))
print("\n  Within the range spanned by the four basal lines?")
for mk, v, lo, hi, ok in in_range:
    print(f"    {mk:<7} {v:+.2f}  (basal lines span {lo:+.2f} to {hi:+.2f})  -> {'YES' if ok else 'no'}")

# Discrete cluster, or the extreme tail of a continuum?
score = S[["CD44", "CD49f"]].mean(1) - S[["CD24", "GATA3"]].mean(1)
print(f"\n=== Discrete population or tail? ===")
print(f"  basal-stem score: consensus {score[MCF7_BASAL].mean():+.2f} vs rest {score[~MCF7_BASAL].mean():+.2f}")
print(f"  consensus cells sit above the {100*(score < score[MCF7_BASAL].min()).mean():.0f}th percentile")
print(f"  cells with score > 1.0: {(score > 1.0).sum():,} ({100*(score > 1.0).mean():.2f}%)")
print("  -> best described as the extreme end of a continuum, not a cleanly separated cluster.")

In [ ]:
# --- 9b.3 The decisive test: where do these cells sit in the JOINT model? ------------------------
# If MCF7's basal-like minority is real, it should occupy the same joint archetypes that the
# BASAL lines occupy -- not MCF7's own luminal archetype.
jsid = pd.read_csv(joint_run / "sample_ids.csv")["sample_id"].to_numpy()
Wj = np.load(joint_run / "W.npy")
dom_j = Wj.argmax(1)[jsid == "MCF7"]

a = pd.Series(dom_j[MCF7_BASAL]).value_counts(normalize=True) * 100
b = pd.Series(dom_j[~MCF7_BASAL]).value_counts(normalize=True) * 100
t = pd.DataFrame({"basal_like_%": a, "other_MCF7_%": b}).fillna(0).sort_index()
t.index = [f"Joint A{i+1}" for i in t.index]
t["enrichment"] = (t["basal_like_%"] / t["other_MCF7_%"].replace(0, np.nan)).round(1)
print("=== Joint-model assignment of MCF7's basal-like cells ===")
print(t.round(1).to_string())

top = t["enrichment"].idxmax()
print(f"\n  Most enriched joint archetype: {top} ({t.loc[top,'enrichment']:.0f}x)")
print("  Joint A5 = the mesenchymal CD44+/CD24- archetype (Vimentin-high) that is 58% SUM149")
print("  Joint A8 = the CD49f+/open-chromatin progenitor archetype, 30% MDA-MB-468")
print("  -> MCF7's minority occupies the SAME states that dominate the basal lines.")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(S.loc[~MCF7_BASAL, "CD24"], S.loc[~MCF7_BASAL, "CD44"], s=1, alpha=0.08,
                color="#bbbbbb", rasterized=True, label="MCF7 bulk")
axes[0].scatter(S.loc[MCF7_BASAL, "CD24"], S.loc[MCF7_BASAL, "CD44"], s=4, alpha=0.7,
                color="#d73027", rasterized=True, label="basal-like consensus")
axes[0].axhline(0, color="k", lw=0.8, ls="--"); axes[0].axvline(0, color="k", lw=0.8, ls="--")
axes[0].set_xlabel("CD24"); axes[0].set_ylabel("CD44")
axes[0].set_title(f"MCF7 CD44/CD24 space\n{MCF7_BASAL.sum():,} cells ({100*MCF7_BASAL.mean():.2f}%) "
                  "in the CD44+/CD24- quadrant", fontweight="bold")
axes[0].legend(markerscale=6, fontsize=9)

m_ = absdf.loc[["CD44", "CD49f", "KRT5", "CD24", "GATA3"]]
m_.plot(kind="bar", ax=axes[1], edgecolor="black", linewidth=0.4,
        color=["#d73027", "#fdae61", "#66c2a5", "#8da0cb", "#a6d854", "#b3b3b3"])
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_ylabel("mean z (shared space)")
axes[1].set_title("MCF7 basal-like vs MCF7 bulk vs the basal lines\n"
                  "CD44/CD49f reach basal range; KRT5 stays low", fontweight="bold")
axes[1].legend(fontsize=8, ncol=2); axes[1].tick_params(axis="x", rotation=0)

t["enrichment"].fillna(0).plot(kind="bar", ax=axes[2], color="#762a83",
                               edgecolor="black", linewidth=0.5)
axes[2].axhline(1, color="k", lw=1, ls="--", label="no enrichment")
axes[2].set_yscale("symlog"); axes[2].set_ylabel("enrichment vs rest of MCF7")
axes[2].set_title("Where the basal-like cells land\nin the joint K=9 model", fontweight="bold")
axes[2].legend(fontsize=9); axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_MCF7_BasalLike_Population.png", dpi=200, bbox_inches="tight")
plt.show()

## 9c. Chromatin state of the MCF7 basal-like population

A specific hypothesis was proposed for this population: **H3K4me1 high, H3K4me3 high, H3K9me2 high,
H4K20me3 low.** It is tested here.

### The confound that has to be removed first

This population sits ~0.34 z **lower on the global chromatin axis** — the same magnitude axis that
consumed two vertices of the joint model (§8). In raw terms *every* histone mark therefore looks
depressed, and any "mark X is low" reading is confounded by overall signal.

The fix is a **composition correction**: express each cell's marks relative to that cell's own mean
histone signal. This separates "which marks dominate this cell's chromatin" from "how much chromatin
signal the cell has overall". H3K9me2 illustrates why it matters — raw it reads −0.300 (apparently
down), composition-corrected it is positive and rises steeply with phenotype purity.

### Evidence hierarchy used

A single gate can be misleading when the population boundary is fuzzy (§9b: mean Jaccard 0.22). So
the hypothesis is tested against, in increasing order of strength:

1. Four independent population definitions (consensus archetype, CD44⁺/CD24⁻ gate, top 1%, top 0.5%)
2. A **dose-response** across all 87,628 MCF7 cells — Spearman correlation with the basal-stem score
3. A **decile trend**, checking monotonicity rather than just a sign
4. **Cross-line replication** in the four basal lines

In [ ]:
# --- 9c.1 Composition correction ------------------------------------------------------------------
HIST_MK = [m for m in MARKERS if m.startswith(("H2A", "H3", "H4")) and m != "H3S28p"]
gh_mcf7 = S[HIST_MK].mean(1)
COMP = S[HIST_MK].sub(gh_mcf7, axis=0)      # each cell's marks relative to its own mean signal

basal_stem_score = S[["CD44", "CD49f"]].mean(1) - S[["CD24", "GATA3"]].mean(1)
HYPOTHESIS = {"H3K4me1": "HIGH", "H3K4me3": "HIGH", "H3K9me2": "HIGH", "H4K20me3": "LOW"}

print(f"Global chromatin signal: basal-like {gh_mcf7[MCF7_BASAL].mean():+.3f} vs rest "
      f"{gh_mcf7[~MCF7_BASAL].mean():+.3f}  (gap {gh_mcf7[MCF7_BASAL].mean()-gh_mcf7[~MCF7_BASAL].mean():+.3f})")
print("-> every mark looks depressed in RAW terms; composition correction removes this.\n")

print("=== Raw vs composition-corrected, for the four hypothesised marks ===")
print(f"{'mark':<10}{'RAW':>10}{'COMPOSITION':>14}   predicted")
for m in HYPOTHESIS:
    raw = S.loc[MCF7_BASAL, m].mean() - S.loc[~MCF7_BASAL, m].mean()
    cmpd = COMP.loc[MCF7_BASAL, m].mean() - COMP.loc[~MCF7_BASAL, m].mean()
    print(f"{m:<10}{raw:>+10.3f}{cmpd:>+14.3f}   {HYPOTHESIS[m]}")

In [ ]:
# --- 9c.2 Robustness across population definitions, then dose-response ---------------------------
defs = {
    "consensus archetype": MCF7_BASAL,
    "CD44+/CD24- gate": ((S["CD44"] > 0) & (S["CD24"] < 0)).values,
    "top 1% score": (basal_stem_score > basal_stem_score.quantile(0.99)).values,
    "top 0.5% score": (basal_stem_score > basal_stem_score.quantile(0.995)).values,
}
print("=== Composition shift under four independent definitions ===")
print(f"{'mark':<11}" + "".join(f"{k:>22}" for k in defs))
for m in HYPOTHESIS:
    print(f"{m:<11}" + "".join(
        f"{COMP.loc[g, m].mean() - COMP.loc[~g, m].mean():>+22.3f}" for g in defs.values()))
print("\n  H3K9me2 strengthens monotonically as the definition tightens -> the consensus set is")
print("  diluted at its boundary, and the weak effect there was an artifact of that dilution.")

from scipy import stats
print("\n=== Dose-response across ALL MCF7 cells (Spearman vs basal-stem score) ===")
rows = []
for m in HIST_MK:
    r, pv = stats.spearmanr(basal_stem_score, COMP[m])
    rows.append({"mark": m, "rho": r, "p": pv,
                 "hypothesis": HYPOTHESIS.get(m, "")})
dose = pd.DataFrame(rows).sort_values("rho", ascending=False).reset_index(drop=True)
print(dose.round(4).to_string(index=False))

dec = pd.qcut(basal_stem_score, 10, labels=False)
trend = COMP.groupby(dec)[list(HYPOTHESIS)].mean()
trend.index = [f"D{i+1}" for i in trend.index]
print("\n=== Decile trend (D1 = least basal-stem, D10 = most) ===")
print(trend.round(3).to_string())
print()
for m in HYPOTHESIS:
    v = trend[m].values
    mono = "monotone UP" if np.all(np.diff(v) > 0) else ("monotone DOWN" if np.all(np.diff(v) < 0)
                                                         else "non-monotone")
    print(f"  {m:<10} D1 {v[0]:+.3f} -> D10 {v[-1]:+.3f}   ({mono})")

print("\n=== VERDICT ===")
print("(a monotone decile trend is required for CONFIRMED; a sign that reverses in the extreme")
print(" deciles is only partial support, since the core of the population contradicts it)\n")
for m, pred in HYPOTHESIS.items():
    r = float(dose.loc[dose["mark"] == m, "rho"].iloc[0])
    v = trend[m].values
    up, down = np.all(np.diff(v) > 0), np.all(np.diff(v) < 0)
    obs = "HIGH" if r > 0 else "LOW"
    if abs(r) < 0.05:
        verdict = "NOT SUPPORTED (no meaningful trend)"
    elif obs != pred:
        verdict = "CONTRADICTED"
    elif (pred == "HIGH" and up) or (pred == "LOW" and down):
        verdict = "CONFIRMED (monotone)"
    else:
        verdict = "PARTIAL (right direction overall, reverses in the extreme deciles)"
    print(f"  {m:<10} predicted {pred:<5} rho={r:+.3f}  D1->D10 {v[0]:+.3f}->{v[-1]:+.3f}  -> {verdict}")

print("\n=== The largest effect was not among the hypothesised marks ===")
top_neg = dose.iloc[-1]
print(f"  {top_neg['mark']} rho={top_neg['rho']:+.3f} -- stronger than H3K9me2 (+{dose.loc[dose['mark']=='H3K9me2','rho'].iloc[0]:.3f}),")
print("  and with H3K27me2 and H2AK119ub also depleted this is a coherent Polycomb-LOSS signature.")
print("  Combined: Polycomb repression down, H3K9me2 heterochromatin up, acetylation up,")
print("  H3K4me1 enhancer priming down -- a switch in repressive mode, not a global gain or loss.")

In [ ]:
# --- 9c.3 Cross-line replication and the Polycomb result -----------------------------------------
print("=== Does the H3K9me2 trend replicate in the basal lines? ===")
rep = []
for line in DISP_ORDER:
    lk = per_line_model[line]["line_key"]
    Sl = pd.DataFrame(line_data[lk]["X"], columns=MARKERS)
    Cl = Sl[HIST_MK].sub(Sl[HIST_MK].mean(1), axis=0)
    sc = Sl[["CD44", "CD49f"]].mean(1) - Sl[["CD24", "GATA3"]].mean(1)
    r9, _ = stats.spearmanr(sc, Cl["H3K9me2"])
    r27, _ = stats.spearmanr(sc, Cl["H3K27me3"])
    rep.append({"line": line, "rho_H3K9me2": r9, "rho_H3K27me3": r27})
rep = pd.DataFrame(rep)
print(rep.round(3).to_string(index=False))
print(f"\n  H3K9me2 positive in {int((rep['rho_H3K9me2']>0).sum())}/5 lines, "
      f"H3K27me3 negative in {int((rep['rho_H3K27me3']<0).sum())}/5 lines")
print("  -> a Polycomb-depleted / H3K9me2-enriched state is a general feature of basal-stem")
print("     character, not an MCF7 peculiarity.")

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

d = dose.sort_values("rho")
cols = ["#d73027" if m in HYPOTHESIS else "#9ecae1" for m in d["mark"]]
axes[0].barh(range(len(d)), d["rho"], color=cols, edgecolor="black", linewidth=0.5)
axes[0].set_yticks(range(len(d))); axes[0].set_yticklabels(d["mark"])
axes[0].axvline(0, color="k", lw=1)
axes[0].set_xlabel("Spearman rho vs basal-stem score")
axes[0].set_title("Chromatin composition dose-response\n(red = hypothesised marks)", fontweight="bold")
axes[0].grid(True, axis="x", linestyle="--", alpha=0.4)

for m, c in zip(HYPOTHESIS, ["#1b9e77", "#d95f02", "#7570b3", "#e7298a"]):
    axes[1].plot(range(1, 11), trend[m], marker="o", lw=2, color=c, label=m)
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_xlabel("Basal-stem score decile (D10 = most basal-like)")
axes[1].set_ylabel("composition-corrected mark level")
axes[1].set_title("Decile trends for the hypothesised marks\nH3K9me2 rises monotonically",
                  fontweight="bold")
axes[1].legend(fontsize=9); axes[1].grid(True, linestyle="--", alpha=0.4)

x = np.arange(len(rep)); w = 0.38
axes[2].bar(x - w/2, rep["rho_H3K9me2"], w, label="H3K9me2", color="#7570b3", edgecolor="black")
axes[2].bar(x + w/2, rep["rho_H3K27me3"], w, label="H3K27me3", color="#66a61e", edgecolor="black")
axes[2].axhline(0, color="k", lw=1)
axes[2].set_xticks(x); axes[2].set_xticklabels(rep["line"], rotation=20)
axes[2].set_ylabel("Spearman rho vs basal-stem score")
axes[2].set_title("Cross-line replication\nH3K9me2 up, Polycomb down, in every line", fontweight="bold")
axes[2].legend(fontsize=9); axes[2].grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(PLOTS / "CyEmbed_MCF7_BasalLike_Chromatin.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Summary

In [ ]:
print("=" * 78)
print("Per-line CyEmbed archetypes and cross-line matching — summary")
print("=" * 78)

print("\n-- Per-line fits (independent, no sample offset, shared scaler) --")
for line in DISP_ORDER:
    m = per_line_model[line]
    s = sel_all[(sel_all.line == line) & (sel_all.K == m["K"])].iloc[0]
    print(f"  {line:<11} n={line_data[m['line_key']]['X'].shape[0]:>7,}  K={m['K']:<3} "
          f"val_recon={s['val_best']:.4f}  cross-seed stability={s['stability']:.3f}")
print(f"  Total archetypes fitted: {TOTAL_ARCH}")

print("\n-- Cross-line matching --")
print(f"  Null threshold (99th pct of marker-permuted cosine) : {MATCH_THRESHOLD:.3f}")
print(f"  Optimal pairings above threshold                    : "
      f"{int(pairs_df['matched'].sum())}/{len(pairs_df)} ({100*pairs_df['matched'].mean():.0f}%)")
print(f"  Most similar line pair                              : "
      f"{pair_scores.astype(float).where(~np.eye(len(DISP_ORDER),dtype=bool)).stack().idxmax()} "
      f"({pair_scores.astype(float).where(~np.eye(len(DISP_ORDER),dtype=bool)).stack().max():.3f})")
print(f"  Least similar line pair                             : "
      f"{pair_scores.astype(float).where(~np.eye(len(DISP_ORDER),dtype=bool)).stack().idxmin()} "
      f"({pair_scores.astype(float).where(~np.eye(len(DISP_ORDER),dtype=bool)).stack().min():.3f})")

print("\n-- Meta-archetypes --")
print(f"  {TOTAL_ARCH} per-line archetypes -> {len(meta_df)} meta-archetypes")
for kind in ["CONSERVED (all 5 lines)", "shared (4 lines)", "shared (3 lines)",
             "restricted (2 lines)", "LINE-SPECIFIC"]:
    g = meta_df[meta_df["kind"] == kind]
    if len(g):
        print(f"    {kind:<24} {len(g):>2}  (metas: {', '.join('M'+str(m) for m in g['meta'])})")

print("\n-- Global chromatin axis --")
print(f"  Reproduced independently in {int((axis_df['vertices_on_axis']>0).sum())}/5 lines; "
      f"mean {axis_df['pct_cells_on_axis'].mean():.0f}% of cells")

print("\n-- Agreement with the joint K=9 model --")
print(f"  {int(cmp_tbl['above_null'].sum())}/{len(cmp_tbl)} joint archetypes have a per-line counterpart")
print("=" * 78)